# 🧬 EchoSense AI — Customer Travel Package Purchase Prediction
## IBM SkillsBuild Final Project — Model Training Notebook

This notebook trains a **Random Forest Classifier** on the `Customertravel.csv` dataset to predict whether a customer will purchase a travel package.

**Dataset**: 954 records | 7 features | Binary Classification (Target: 0/1)

---

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc
import warnings
warnings.filterwarnings('ignore')

sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (10, 6)
print('All libraries imported successfully!')

## 2. Load Dataset

In [ ]:
df = pd.read_csv('Customertravel.csv')
print(f'Dataset Shape: {df.shape}')
print(f'Total Records: {df.shape[0]}')
print(f'Total Features: {df.shape[1]}')
df.head(10)

## 3. Data Overview & Info

In [ ]:
print('=== Dataset Info ===')
print(df.info())
print('\n=== Statistical Summary ===')
df.describe()

In [ ]:
print('=== Missing Values ===')
print(df.isnull().sum())
print(f'\nTotal missing values: {df.isnull().sum().sum()}')
print('\n=== Unique Values Per Column ===')
for col in df.columns:
    print(f'{col}: {df[col].nunique()} unique values -> {df[col].unique()}')

## 4. Exploratory Data Analysis (EDA)
### 4.1 Target Variable Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
target_counts = df['Target'].value_counts()
colors = ['#333333', '#8A3FFC']
axes[0].bar(target_counts.index.map({0: 'Not Purchased', 1: 'Purchased'}), 
            target_counts.values, color=colors)
axes[0].set_title('Target Distribution (Count)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontweight='bold', fontsize=12)

# Pie chart
axes[1].pie(target_counts.values, labels=['Not Purchased (0)', 'Purchased (1)'],
            colors=colors, autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Target Distribution (%)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()
print(f'Class 0 (Not Purchased): {target_counts[0]} ({target_counts[0]/len(df)*100:.1f}%)')
print(f'Class 1 (Purchased): {target_counts[1]} ({target_counts[1]/len(df)*100:.1f}%)')

### 4.2 Age Distribution by Target

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
df[df['Target']==0]['Age'].hist(alpha=0.6, bins=15, color='#333333', label='Not Purchased', ax=ax)
df[df['Target']==1]['Age'].hist(alpha=0.7, bins=15, color='#8A3FFC', label='Purchased', ax=ax)
ax.set_title('Age Distribution by Purchase Status', fontsize=14, fontweight='bold')
ax.set_xlabel('Age')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.show()

### 4.3 Categorical Feature Analysis

In [ ]:
cat_features = ['FrequentFlyer', 'AnnualIncomeClass', 'AccountSyncedToSocialMedia', 'BookedHotelOrNot']

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for idx, col in enumerate(cat_features):
    ax = axes[idx // 2][idx % 2]
    ct = pd.crosstab(df[col], df['Target'])
    ct.plot(kind='bar', ax=ax, color=['#333333', '#8A3FFC'], width=0.7)
    ax.set_title(f'{col} vs Target', fontsize=13, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
    ax.legend(['Not Purchased', 'Purchased'])
    ax.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

### 4.4 Services Opted Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ct_services = pd.crosstab(df['ServicesOpted'], df['Target'])
ct_services.plot(kind='bar', ax=ax, color=['#333333', '#8A3FFC'], width=0.7)
ax.set_title('Services Opted vs Purchase Decision', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Services Opted')
ax.set_ylabel('Count')
ax.legend(['Not Purchased', 'Purchased'])
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

### 4.5 Correlation Heatmap

In [ ]:
df_encoded = df.copy()
for col in cat_features:
    df_encoded[col] = LabelEncoder().fit_transform(df_encoded[col])

fig, ax = plt.subplots(figsize=(10, 8))
corr_matrix = df_encoded.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdPu',
            center=0, square=True, linewidths=1, ax=ax)
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Data Preprocessing
### 5.1 Encode Categorical Variables

In [ ]:
df_processed = df.copy()

categorical_cols = ['FrequentFlyer', 'AnnualIncomeClass', 'AccountSyncedToSocialMedia', 'BookedHotelOrNot']
le_dict = {}

for col in categorical_cols:
    le = LabelEncoder()
    df_processed[col] = le.fit_transform(df_processed[col])
    le_dict[col] = le
    print(f'{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}')

print('\nEncoded DataFrame:')
df_processed.head()

### 5.2 Split Features & Target

In [ ]:
X = df_processed.drop('Target', axis=1)
y = df_processed['Target']

print(f'Feature Matrix Shape: {X.shape}')
print(f'Target Vector Shape: {y.shape}')
print(f'\nFeature Names: {X.columns.tolist()}')

### 5.3 Feature Scaling (StandardScaler)

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Scaled Feature Statistics:')
print(f'Mean (should be ~0): {X_scaled.mean(axis=0).round(4)}')
print(f'Std  (should be ~1): {X_scaled.std(axis=0).round(4)}')

### 5.4 Train-Test Split (80/20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

print(f'Training Set: {X_train.shape[0]} samples')
print(f'Testing Set:  {X_test.shape[0]} samples')
print(f'\nTraining Target Distribution:\n{y_train.value_counts()}')
print(f'\nTesting Target Distribution:\n{y_test.value_counts()}')

## 6. Model Training — Random Forest Classifier

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

print('Random Forest Classifier trained successfully!')
print(f'Number of Trees: {rf.n_estimators}')
print(f'Max Depth: {rf.max_depth}')
print(f'Criterion: {rf.criterion}')

## 7. Model Evaluation
### 7.1 Accuracy & Classification Report

In [ ]:
y_pred = rf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f'Model Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)')
print(f'\n{"="*50}')
print('Classification Report:')
print(f'{"="*50}')
print(classification_report(y_test, y_pred, target_names=['Not Purchased', 'Purchased']))

### 7.2 Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', 
            xticklabels=['Not Purchased', 'Purchased'],
            yticklabels=['Not Purchased', 'Purchased'], ax=ax,
            annot_kws={'size': 16})
ax.set_title('Confusion Matrix', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
plt.tight_layout()
plt.show()

print(f'True Negatives:  {cm[0][0]}')
print(f'False Positives: {cm[0][1]}')
print(f'False Negatives: {cm[1][0]}')
print(f'True Positives:  {cm[1][1]}')

### 7.3 ROC Curve

In [ ]:
y_prob = rf.predict_proba(X_test)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr, tpr, color='#8A3FFC', lw=2, label=f'ROC Curve (AUC = {roc_auc:.4f})')
ax.plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1)
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=12)
plt.tight_layout()
plt.show()

print(f'AUC Score: {roc_auc:.4f}')

### 7.4 Feature Importance

In [ ]:
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(feature_importance['Feature'], feature_importance['Importance'], color='#8A3FFC')
ax.set_title('Feature Importance (Random Forest)', fontsize=14, fontweight='bold')
ax.set_xlabel('Importance Score')
for i, v in enumerate(feature_importance['Importance']):
    ax.text(v + 0.002, i, f'{v:.4f}', va='center', fontsize=10)
plt.tight_layout()
plt.show()

print('Feature Importance Ranking:')
for _, row in feature_importance.sort_values('Importance', ascending=False).iterrows():
    print(f"  {row['Feature']:35s} -> {row['Importance']:.4f}")

## 8. Save Model Artifacts

In [ ]:
artifacts = {
    'model': rf,
    'scaler': scaler,
    'le_dict': le_dict,
    'feature_names': X.columns.tolist(),
    'categorical_cols': categorical_cols
}

joblib.dump(artifacts, 'travel_model_assets.pkl')

import os
model_size = os.path.getsize('travel_model_assets.pkl')
print(f'Model saved to: travel_model_assets.pkl')
print(f'File size: {model_size / (1024*1024):.2f} MB')
print(f'\nArtifacts saved:')
print(f'  - Trained RandomForestClassifier (100 trees)')
print(f'  - StandardScaler')
print(f'  - LabelEncoder dictionary ({len(le_dict)} encoders)')
print(f'  - Feature names: {X.columns.tolist()}')

## 9. Test Model Loading & Inference

In [ ]:
# Load saved model and test inference
loaded = joblib.load('travel_model_assets.pkl')

# Sample prediction
sample = pd.DataFrame([{
    'Age': 30,
    'FrequentFlyer': 'Yes',
    'AnnualIncomeClass': 'High Income',
    'ServicesOpted': 4,
    'AccountSyncedToSocialMedia': 'No',
    'BookedHotelOrNot': 'Yes'
}])

for col, le in loaded['le_dict'].items():
    sample[col] = le.transform(sample[col])

X_sample = loaded['scaler'].transform(sample[loaded['feature_names']])
prob = loaded['model'].predict_proba(X_sample)[0][1]

print(f'Sample Customer Prediction:')
print(f'  Purchase Probability: {prob*100:.1f}%')
print(f'  Prediction: {"Will Purchase" if prob > 0.5 else "Will Not Purchase"}')

---
## 10. Conclusion

- **Dataset**: 954 customer records with 6 features
- **Model**: Random Forest Classifier (100 trees)
- **Accuracy**: ~88%
- **Key Findings**:
  - Age and FrequentFlyer status are strong predictors
  - High Income customers show higher purchase tendency
  - Number of services opted correlates with purchase likelihood
- **Model saved** as `travel_model_assets.pkl` for deployment in the EchoSense AI Streamlit dashboard

---
*EchoSense AI | IBM SkillsBuild Final Project | 2026*